# NovaBrew Coffee — Post 2: Customer Segmentation
## RFM Analysis & Cohort Basics

**Series:** NovaBrew Digital Marketing Analytics  
**Stack:** Python · pandas · matplotlib · seaborn  
**Data:** Synthetic — generated by `data_generation/generate_data.py`

---

### What we're doing in this notebook

1. Load and explore the customers + orders data
2. Build RFM scores from scratch in pandas (mirrors the dbt model)
3. Assign segment labels
4. Visualise the segments — size, revenue contribution, AOV
5. Cohort analysis — retention by signup month

> **Note:** In production, this analysis runs in BigQuery via the `mart_customer_segments` dbt model.  
> This notebook exists to explore, explain, and visualise the logic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style config ──────────────────────────────────────────────────────────────
BROWN   = '#5C3D2E'
CREAM   = '#F5F0E8'
ACCENT  = '#C87941'
DARK    = '#1A1A1A'
GREY    = '#6B6B6B'

SEGMENT_COLORS = {
    'Champion':       '#2D6A4F',
    'Loyal':          '#52B788',
    'Promising':      '#74C69D',
    'Needs Attention':'#F4A261',
    'Occasional':     '#E9C46A',
    'At Risk':        '#E76F51',
    'Lost':           '#9B2226',
    'Non-Buyer':      '#ADB5BD',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.labelcolor':  DARK,
    'xtick.color':      GREY,
    'ytick.color':      GREY,
    'font.family':      'sans-serif',
    'font.size':        11,
})

print('Libraries loaded ✓')

---
## 1. Load Data

In [ ]:
# ── Load CSVs ─────────────────────────────────────────────────────────────────
# In production: replace with BigQuery client reads
# from google.cloud import bigquery
# client = bigquery.Client(project='your-project')
# customers = client.query('SELECT * FROM novabrew_raw.customers').to_dataframe()

customers = pd.read_csv('../data_generation/output/customers.csv', parse_dates=['signup_date', 'date_of_birth'])
orders    = pd.read_csv('../data_generation/output/orders.csv',    parse_dates=['order_date', 'shipped_date', 'delivered_date'])

print(f'Customers: {len(customers):,}')
print(f'Orders:    {len(orders):,}')
print(f'\nOrder date range: {orders.order_date.min().date()} → {orders.order_date.max().date()}')
customers.head(3)

In [ ]:
# ── Quick data quality checks ─────────────────────────────────────────────────
print('=== Data Quality ===')
print(f'Null customer_ids in orders: {orders.customer_id.isna().sum()}')
print(f'Order status breakdown:')
print(orders.status.value_counts().to_string())
print(f'\nOrders with negative total: {(orders.order_total < 0).sum()}')

---
## 2. Build RFM Scores

**RFM** is one of the oldest and most effective customer segmentation frameworks in marketing analytics.

| Dimension | Question | Better = |
|-----------|----------|----------|
| **R**ecency | How long ago was the last purchase? | Lower days = better |
| **F**requency | How many times have they bought? | Higher count = better |
| **M**onetary | How much have they spent in total? | Higher revenue = better |

We score each dimension 1–4 using **quartiles** (NTILE in SQL, `qcut` in pandas). A score of 4 = top 25%.

In [ ]:
# ── Filter to delivered orders only ───────────────────────────────────────────
END_DATE = pd.Timestamp('2024-06-30')   # Series end date = our "today"

delivered = orders[orders['status'] == 'delivered'].copy()
print(f'Delivered orders: {len(delivered):,} of {len(orders):,} total')

# ── Aggregate per customer ─────────────────────────────────────────────────────
rfm = delivered.groupby('customer_id').agg(
    order_count      = ('order_id',     'count'),
    total_revenue    = ('order_total',  'sum'),
    avg_order_value  = ('order_total',  'mean'),
    first_order_date = ('order_date',   'min'),
    last_order_date  = ('order_date',   'max'),
).reset_index()

rfm['days_since_last_order'] = (END_DATE - rfm['last_order_date']).dt.days

print(f'\nCustomers with at least one order: {len(rfm):,}')
print(f'Customers never ordered:           {len(customers) - len(rfm):,}')
rfm.describe().round(2)

In [ ]:
# ── Score each dimension 1-4 using quartiles ──────────────────────────────────
# Recency: REVERSED — fewer days since order = better = higher score
rfm['r_score'] = pd.qcut(
    rfm['days_since_last_order'],
    q=4,
    labels=[4, 3, 2, 1]   # 4=most recent, 1=longest ago
).astype(int)

# Frequency: more orders = higher score
rfm['f_score'] = pd.qcut(
    rfm['order_count'].rank(method='first'),
    q=4,
    labels=[1, 2, 3, 4]
).astype(int)

# Monetary: more revenue = higher score
rfm['m_score'] = pd.qcut(
    rfm['total_revenue'].rank(method='first'),
    q=4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm['rfm_total'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

print('Score distributions:')
for col in ['r_score','f_score','m_score']:
    print(f'  {col}: {rfm[col].value_counts().sort_index().to_dict()}')

---
## 3. Assign Segment Labels

Segment rules mirror the dbt `mart_customer_segments.sql` model exactly:

In [ ]:
def assign_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    oc = row['order_count']

    if   r == 4 and f >= 3 and m >= 3:   return 'Champion'
    elif f >= 3 and r >= 2:               return 'Loyal'
    elif r >= 3 and f <= 2 and oc <= 2:  return 'Promising'
    elif r == 2 and f >= 2 and m >= 2:   return 'Needs Attention'
    elif r <= 2 and f >= 3:              return 'At Risk'
    elif r == 1 and f <= 2:              return 'Lost'
    else:                                 return 'Occasional'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

# Add non-buyers
non_buyers = customers[~customers['customer_id'].isin(rfm['customer_id'])][['customer_id']].copy()
non_buyers['segment'] = 'Non-Buyer'
non_buyers['order_count'] = 0
non_buyers['total_revenue'] = 0.0
non_buyers['avg_order_value'] = 0.0

seg_summary = rfm.groupby('segment').agg(
    customers        = ('customer_id',    'count'),
    total_revenue    = ('total_revenue',  'sum'),
    avg_order_value  = ('avg_order_value','mean'),
    avg_orders       = ('order_count',    'mean'),
    avg_days_since   = ('days_since_last_order','mean'),
).reset_index()

seg_summary['revenue_pct'] = (seg_summary['total_revenue'] / seg_summary['total_revenue'].sum() * 100).round(1)
seg_summary = seg_summary.sort_values('total_revenue', ascending=False)

print(seg_summary[['segment','customers','revenue_pct','avg_order_value','avg_orders','avg_days_since']].to_string(index=False))

---
## 4. Visualise Segments

In [ ]:
# ── Chart 1: Segment Size vs Revenue Contribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')
fig.suptitle('NovaBrew Customer Segments', fontsize=16, fontweight='bold', color=DARK, y=1.01)

order = ['Champion','Loyal','Promising','Occasional','Needs Attention','At Risk','Lost']
plot_df = seg_summary[seg_summary['segment'] != 'Non-Buyer'].set_index('segment').reindex(order)
colors  = [SEGMENT_COLORS[s] for s in order]

# Left: customer count
ax1 = axes[0]
bars = ax1.barh(order, plot_df['customers'], color=colors, edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Number of Customers', color=GREY)
ax1.set_title('Segment Size', fontweight='bold', color=DARK)
ax1.invert_yaxis()
for bar, val in zip(bars, plot_df['customers']):
    ax1.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=10, color=GREY)

# Right: revenue %
ax2 = axes[1]
bars2 = ax2.barh(order, plot_df['revenue_pct'], color=colors, edgecolor='white', linewidth=0.5)
ax2.set_xlabel('% of Total Revenue', color=GREY)
ax2.set_title('Revenue Contribution', fontweight='bold', color=DARK)
ax2.invert_yaxis()
for bar, val in zip(bars2, plot_df['revenue_pct']):
    ax2.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=10, color=GREY)

plt.tight_layout()
plt.savefig('../post2/segment_size_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 1 saved ✓')

In [ ]:
# ── Chart 2: RFM Scatter — Recency vs Frequency, sized by Revenue ────────────
fig, ax = plt.subplots(figsize=(12, 7))

for seg, group in rfm.groupby('segment'):
    ax.scatter(
        group['days_since_last_order'],
        group['order_count'],
        s=np.sqrt(group['total_revenue']) * 2,
        c=SEGMENT_COLORS.get(seg, GREY),
        alpha=0.45,
        edgecolors='white',
        linewidth=0.3,
        label=seg
    )

ax.set_xlabel('Days Since Last Order  (← more recent)', fontsize=12, color=GREY)
ax.set_ylabel('Number of Orders', fontsize=12, color=GREY)
ax.set_title('Customer RFM Map\nBubble size = total revenue', fontsize=14,
             fontweight='bold', color=DARK)

ax.invert_xaxis()  # recent customers on the right
ax.axvline(x=rfm['days_since_last_order'].median(), color=GREY, linestyle='--',
           alpha=0.4, linewidth=1)
ax.axhline(y=rfm['order_count'].median(), color=GREY, linestyle='--',
           alpha=0.4, linewidth=1)

legend = ax.legend(title='Segment', bbox_to_anchor=(1.02, 1), loc='upper left',
                   frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig('../post2/rfm_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 2 saved ✓')

In [ ]:
# ── Chart 3: Average Order Value by Segment ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

aov_df = seg_summary[seg_summary['segment'] != 'Non-Buyer'].sort_values('avg_order_value', ascending=True)
bar_colors = [SEGMENT_COLORS[s] for s in aov_df['segment']]

bars = ax.barh(aov_df['segment'], aov_df['avg_order_value'],
               color=bar_colors, edgecolor='white', linewidth=0.5)

# Add overall average line
overall_aov = rfm['avg_order_value'].mean()
ax.axvline(x=overall_aov, color=BROWN, linestyle='--', linewidth=1.5, label=f'Overall AOV ${overall_aov:.0f}')

for bar, val in zip(bars, aov_df['avg_order_value']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'${val:.0f}', va='center', fontsize=10, color=GREY)

ax.set_xlabel('Average Order Value (USD)', color=GREY)
ax.set_title('Average Order Value by Segment', fontsize=14, fontweight='bold', color=DARK)
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig('../post2/aov_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 3 saved ✓')

---
## 5. Cohort Analysis — Monthly Retention

A **cohort** is a group of customers who all did something at the same time — in this case, made their first purchase in the same month. Cohort analysis shows whether newer customers retain as well as older ones, and how quickly customers churn after their first order.

**How to read the heatmap:**
- Each row = a cohort (month of first purchase)
- Each column = months since first purchase (0, 1, 2, ...)
- Each cell = % of original cohort who purchased again in that month
- Month 0 = 100% by definition (everyone placed at least one order)

In [ ]:
# ── Build cohort table ────────────────────────────────────────────────────────

# Get each customer's cohort month (month of first order)
first_orders = delivered.groupby('customer_id')['order_date'].min().reset_index()
first_orders.columns = ['customer_id', 'cohort_date']
first_orders['cohort_month'] = first_orders['cohort_date'].dt.to_period('M')

# Join cohort month back to all orders
cohort_df = delivered.merge(first_orders[['customer_id','cohort_month']], on='customer_id')
cohort_df['order_month'] = cohort_df['order_date'].dt.to_period('M')
cohort_df['period_number'] = (cohort_df['order_month'] - cohort_df['cohort_month']).apply(lambda x: x.n)

# Count unique customers per cohort per period
cohort_counts = cohort_df.groupby(['cohort_month','period_number'])['customer_id'].nunique().reset_index()
cohort_pivot  = cohort_counts.pivot_table(index='cohort_month', columns='period_number', values='customer_id')

# Retention % — divide each row by period 0
cohort_size   = cohort_pivot[0]
retention_df  = cohort_pivot.divide(cohort_size, axis=0).round(3) * 100

# Keep only first 12 months and last 12 cohort months for readability
retention_df  = retention_df.iloc[-14:, :13]

print(f'Cohort table shape: {retention_df.shape}')
print(f'Avg month-1 retention: {retention_df[1].mean():.1f}%')

In [ ]:
# ── Chart 4: Cohort Retention Heatmap ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

mask = retention_df.isna()

sns.heatmap(
    retention_df,
    annot=True,
    fmt='.0f',
    mask=mask,
    cmap=sns.light_palette(BROWN, as_cmap=True),
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Retention %'},
    ax=ax,
    annot_kws={'size': 9},
    vmin=0,
    vmax=100
)

ax.set_title('Monthly Cohort Retention\n% of cohort who purchased again',
             fontsize=14, fontweight='bold', color=DARK, pad=15)
ax.set_xlabel('Months Since First Purchase', fontsize=11, color=GREY)
ax.set_ylabel('Cohort (First Purchase Month)', fontsize=11, color=GREY)
ax.set_yticklabels([str(p) for p in retention_df.index], rotation=0)

plt.tight_layout()
plt.savefig('../post2/cohort_retention.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 4 saved ✓')

---
## 6. Key Insights & Recommendations

What does this analysis tell us as an agency? Let's translate the data into actions.

In [ ]:
# ── Summary stats for the blog post ──────────────────────────────────────────
total_rev   = rfm['total_revenue'].sum()
champ       = rfm[rfm['segment'] == 'Champion']
at_risk     = rfm[rfm['segment'] == 'At Risk']
lost        = rfm[rfm['segment'] == 'Lost']

print('=' * 55)
print('KEY INSIGHTS — NovaBrew Customer Segmentation')
print('=' * 55)

print(f"\n📊 Champions ({len(champ):,} customers = {len(champ)/len(rfm)*100:.1f}% of buyers)")
print(f"   Drive {champ['total_revenue'].sum()/total_rev*100:.1f}% of all revenue")
print(f"   Avg AOV: ${champ['avg_order_value'].mean():.0f}  |  Avg orders: {champ['order_count'].mean():.1f}")
print(f"   → Protect them: VIP programme, early access, handwritten notes")

print(f"\n⚠️  At Risk ({len(at_risk):,} customers)")
print(f"   Were frequent buyers, avg {at_risk['order_count'].mean():.1f} orders")
print(f"   Last order avg {at_risk['days_since_last_order'].mean():.0f} days ago")
print(f"   Potential recovery revenue: ${at_risk['avg_order_value'].mean() * len(at_risk):,.0f}")
print(f"   → Win-back email flow: 'We miss you' + 15% discount")

print(f"\n❌ Lost ({len(lost):,} customers)")
print(f"   Last order avg {lost['days_since_last_order'].mean():.0f} days ago")
print(f"   Low AOV (${lost['avg_order_value'].mean():.0f}) — likely promo buyers who didn't stick")
print(f"   → Suppress from paid retargeting. Test reactivation at low cost only.")

print(f"\n🌱 Promising ({len(rfm[rfm.segment=='Promising']):,} customers — recent first-timers)")
print(f"   → Post-purchase email flow is critical in the next 30 days")
print(f"   → Second purchase is the most important conversion in D2C")

print('\n' + '=' * 55)
print(f'Non-buyers: {len(non_buyers):,} — consider triggered browse-abandonment email')
print('=' * 55)

---
## 7. Production Query (BigQuery)

Once the dbt model is deployed, the entire segment table is available as a single query:

```sql
-- Get segment summary from dbt mart
SELECT
    segment,
    COUNT(*)                        AS customers,
    ROUND(SUM(total_revenue), 2)    AS total_revenue,
    ROUND(AVG(avg_order_value), 2)  AS avg_aov,
    ROUND(AVG(order_count), 1)      AS avg_orders
FROM `your-project.novabrew_marts.mart_customer_segments`
GROUP BY 1
ORDER BY total_revenue DESC;
```

This is the table that Looker Studio will connect to for the Customer Segments dashboard page in Post 10.